In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

import matplotlib.pyplot as plt

import sys
import os

sys.path.append(os.path.abspath(".."))

In [27]:
amazon="..\\data\\raw\\Reviews.csv"
tweet="..\\data\\raw\\training.1600000.processed.noemoticon.csv"

### Preprocessing of amazon dataset

In [28]:
#loading the data
df_amazon = pd.read_csv(amazon)

In [29]:
df_amazon.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [30]:
df_amazon = df_amazon[["Text", "Score"]]

# Binary sentiment
df_amazon["label"] = df_amazon["Score"].apply(
    lambda x: 1 if x >= 4 else 0
)

df_amazon = df_amazon[["Text", "label"]]

df_amazon = df_amazon.dropna()

df_amazon = df_amazon.sample(10000, random_state=42)

df_amazon.head()

,Text,label
165256,Having tried a couple of other brands of glute...,1
231465,My cat loves these treats. If ever I can't fin...,1
427827,A little less than I expected. It tends to ha...,0
433954,"First there was Frosted Mini-Wheats, in origin...",0
70260,and I want to congratulate the graphic artist ...,1


### preprocessing of tweets dataset

In [31]:
twitter_df = pd.read_csv(
    tweet,
    encoding="latin-1",
    header=None
)

In [32]:
twitter_df.head()

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [33]:
twitter_df = twitter_df[[0, 5]]

twitter_df.columns = ["label", "Text"]

twitter_df["label"] = twitter_df["label"].map({
    0: 0,
    4: 1
})

twitter_df = twitter_df.dropna()

twitter_df = twitter_df.sample(10000, random_state=42)

twitter_df.head()

,label,Text
541200,0,@chrishasboobs AHHH I HOPE YOUR OK!!!
750,0,"@misstoriblack cool , i have no tweet apps fo..."
766711,0,@TiannaChaos i know just family drama. its la...
285055,0,School email won't open and I have geography ...
705995,0,upper airways problem


### Comparing processed and raw text

##### 1)Amazon

In [34]:
from src.preprocessing.textpreprocessing import (
    TextPreprocessor,
    PreprocessingConfig
)

processor = TextPreprocessor(
    PreprocessingConfig.for_reviews()
)

# preprocess text column
df = processor.process_dataframe(
    df=df_amazon,
    text_col="Text",
    out_col="clean_text"
)

print(df[["Text", "clean_text"]].head())

[food_reviews] preprocessing: 100%|██████████| 10000/10000 [00:07<00:00, 1424.66it/s]

                                                     Text  \
165256  Having tried a couple of other brands of glute...   
231465  My cat loves these treats. If ever I can't fin...   
427827  A little less than I expected.  It tends to ha...   
433954  First there was Frosted Mini-Wheats, in origin...   
70260   and I want to congratulate the graphic artist ...   

                                               clean_text  
165256  tried couple brand glutenfree sandwich cooky b...  
231465  cat love treat ever not find house pop top bol...  
427827  little less expected tends muddy taste not exp...  
433954  first frosted miniwheats original size frosted...  
70260   want congratulate graphic artist putting entir...  


##### 2)tweets

In [35]:
from src.preprocessing.textpreprocessing import (
    TextPreprocessor,
    PreprocessingConfig
)

processor = TextPreprocessor(
    PreprocessingConfig.for_tweets()
)

# preprocess text column
df = processor.process_dataframe(
    df=twitter_df,
    text_col="Text",
    out_col="clean_text"
)

print(df[["Text", "clean_text"]].head())

[twitter] preprocessing: 100%|██████████| 10000/10000 [00:01<00:00, 6976.12it/s]
52 rows produced empty strings after preprocessing.


                                                     Text  \
541200             @chrishasboobs AHHH I HOPE YOUR OK!!!    
750     @misstoriblack cool , i have no tweet apps  fo...   
766711  @TiannaChaos i know  just family drama. its la...   
285055  School email won't open  and I have geography ...   
705995                             upper airways problem    

                                               clean_text  
541200                                          ahhh hope  
750                                  cool tweet apps razr  
766711  know family drama lamehey next time hang kim g...  
285055  school email not open geography stuff revise s...  
705995                               upper airway problem  


### Comparing different vectorization methods


#### 1) BOW

In [36]:
from src.vectorization.bow import BoWVectorizer
bow_vectorizer = BoWVectorizer(text_processor=processor)
X_amazon_bow = bow_vectorizer.fit_transform(df_amazon["clean_text"])
X_twitter_bow = bow_vectorizer.fit_transform(twitter_df["clean_text"])

In [37]:
print(X_amazon_bow.shape)
print(X_twitter_bow.shape)

(10000, 5000)
(10000, 5000)


#### 2)TF-IDF

In [38]:
from src.vectorization.tfidf import TFIDFVectorizer

tfidf_vectorizer = TFIDFVectorizer(text_processor=processor)
X_amazon_tfidf = tfidf_vectorizer.fit_transform(df_amazon["clean_text"])
X_twitter_tfidf = tfidf_vectorizer.fit_transform(twitter_df["clean_text"])
print(X_amazon_tfidf.shape)
print(X_twitter_tfidf.shape)

(10000, 5000)
(10000, 5000)


#### 3)bm-25

In [39]:
from src.vectorization.bm25 import BM25Vectorizer
bm25_vectorizer = BM25Vectorizer(text_processor=processor)
X_amazon_bm25 = bm25_vectorizer.fit(df_amazon["clean_text"])
X_twitter_bm25 = bm25_vectorizer.fit(twitter_df["clean_text"])

In [40]:
X_amazon_bm25.search("This product is great!", top_k=5)

(array([9127, 2358, 7898,    9,  123]),
 array([9.62166499, 6.64468248, 6.26701458, 6.25967173, 6.25967173]))

#### Embeddings

##### 1)Word2vec

In [ ]:
from src.vectorization.embeddings import Embeddings
embeddings_vectorizer = Embeddings(embedding_type="word2vec",embedding_path="word2vec\GoogleNews-vectors-negative300.bin")
model = Embeddings.load_embeddings()
print(model["king"])

TypeError: Embeddings.load_embeddings() missing 1 required positional argument: 'self'

##### 2)Glove vectorizer